# 06 · Emotion classification — fine-tuned vs frozen-features baseline

Primary-task pivot (issue #61): 6-class emotion on `dair-ai/emotion` (`split`),
fine-tuning the **task-agnostic** backbone `cardiffnlp/twitter-roberta-base`
(ADR 0011) with balanced class weights (ADR 0012). The fine-tuned model is
measured against a **frozen-features baseline** (backbone embeddings + Logistic
Regression) — the "before fine-tuning" reference it must beat. This replaces the
sentiment thesis disproven in #59 (the old base was already TweetEval-tuned).

In [ ]:
import json
import os
import sys

# Run from the repo root whether launched via `jupyter` (root) or `nbconvert` (notebooks/).
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import DatasetDict
from sklearn.metrics import precision_recall_curve
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer, set_seed

from src.baseline import extract_features, fit_baseline, predict_baseline
from src.evaluation import divergent_classes, evaluation_report, macro_f1_pct_gain, per_class_f1, predict_split
from src.training import DEFAULT_OUTPUT_DIR, LABEL_NAMES, MODEL_NAME, SEED, load_emotion_dataset, tokenize_dataset, train

set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
summary = {}


def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def expected_calibration_error(probs, labels, n_bins=10):
    conf = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == labels).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if mask.sum() > 0:
            ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return float(ece)


def reliability_points(probs, labels, n_bins=10):
    conf = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == labels).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    xs, ys = [], []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if mask.sum() > 0:
            xs.append(float(conf[mask].mean()))
            ys.append(float(correct[mask].mean()))
    return np.array(xs), np.array(ys)


print("device:", DEVICE)

## 1. Dataset and class distribution

In [ ]:
dataset = load_emotion_dataset()
y_train = np.array(dataset["train"]["label"])
y_test = np.array(dataset["test"]["label"])

train_counts = np.bincount(y_train, minlength=len(LABEL_NAMES))
train_dist = {LABEL_NAMES[i]: int(train_counts[i]) for i in range(len(LABEL_NAMES))}
surprise_share = float(train_counts[LABEL_NAMES.index("surprise")]) / float(train_counts.sum())
summary["train_distribution"] = train_dist
summary["surprise_share_pct"] = round(surprise_share * 100, 2)
print("split sizes:", {k: len(v) for k, v in dataset.items()})
print("train distribution:", train_dist)
print("surprise share of train: {:.2f}%".format(surprise_share * 100))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(LABEL_NAMES, train_counts, color="steelblue")
ax.set_title("dair-ai/emotion - train class distribution")
ax.set_ylabel("count")
plt.show()

## 2. Token-length sanity check (`max_length=128`)

In [ ]:
length_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
sample = dataset["train"].select(range(2000))["text"]
lengths = np.array([len(length_tokenizer(t)["input_ids"]) for t in sample])
summary["token_len_p99_sample2000"] = int(np.percentile(lengths, 99))
summary["token_len_max_sample2000"] = int(lengths.max())
print("token length over 2000 train rows: p99={}, max={} (MAX_LENGTH=128)".format(int(np.percentile(lengths, 99)), int(lengths.max())))

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(lengths, bins=40, color="steelblue")
ax.axvline(128, color="red", linestyle="--", label="max_length=128")
ax.set_xlabel("token length")
ax.legend()
plt.show()

## 3. Frozen-features baseline (backbone embeddings + LogisticRegression)

In [ ]:
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

X_train = extract_features(dataset["train"]["text"], base_tokenizer, base_model, batch_size=64)
X_test = extract_features(dataset["test"]["text"], base_tokenizer, base_model, batch_size=64)
print("features:", X_train.shape, X_test.shape)

baseline_clf = fit_baseline(X_train, y_train, seed=SEED)
baseline_logits = predict_baseline(baseline_clf, X_test)
baseline_report = evaluation_report(baseline_logits, y_test)
print("baseline  acc={:.4f}  macroF1={:.4f}".format(baseline_report["accuracy"], baseline_report["f1_macro"]))

del base_model
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 4. Fine-tune with balanced class weights (ADR 0012)

In [ ]:
train(output_dir=DEFAULT_OUTPUT_DIR, use_class_weights=True)

ft_tokenizer = AutoTokenizer.from_pretrained(DEFAULT_OUTPUT_DIR)
ft_model = AutoModelForSequenceClassification.from_pretrained(DEFAULT_OUTPUT_DIR)
test_tokenized = tokenize_dataset(DatasetDict({"test": dataset["test"]}), ft_tokenizer)["test"]
ft_logits = predict_split(ft_model, test_tokenized, batch_size=64)
ft_probs = softmax(ft_logits)
ft_report = evaluation_report(ft_logits, y_test)
print("finetuned acc={:.4f}  macroF1={:.4f}".format(ft_report["accuracy"], ft_report["f1_macro"]))

del ft_model
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 5. Headline — baseline vs fine-tuned (test split)

In [ ]:
gain = macro_f1_pct_gain(baseline_report["f1_macro"], ft_report["f1_macro"])
summary["baseline"] = {"accuracy": round(baseline_report["accuracy"], 4), "f1_macro": round(baseline_report["f1_macro"], 4)}
summary["finetuned"] = {"accuracy": round(ft_report["accuracy"], 4), "f1_macro": round(ft_report["f1_macro"], 4)}
summary["macro_f1_gain_pct"] = round(gain, 2)

table = pd.DataFrame(
    {
        "Model": ["Frozen-features baseline", "Fine-tuned"],
        "Accuracy": [baseline_report["accuracy"], ft_report["accuracy"]],
        "Macro F1": [baseline_report["f1_macro"], ft_report["f1_macro"]],
    }
)
print(table.to_string(index=False))
print("\nMacro-F1 gain of fine-tuned over baseline: {:+.2f}%".format(gain))
beats = bool(ft_report["f1_macro"] > baseline_report["f1_macro"])
summary["finetuned_beats_baseline"] = beats
print("HEADLINE:", "fine-tuned EXCEEDS baseline" if beats else "fine-tuned does NOT exceed baseline")

## 6. Confusion matrix (fine-tuned)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(ft_report["confusion_matrix"], annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Fine-tuned confusion matrix (test)")
plt.show()

## 7. Per-class precision-recall (fine-tuned)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for c in range(len(LABEL_NAMES)):
    precision, recall, _ = precision_recall_curve((y_test == c).astype(int), ft_probs[:, c])
    ax.plot(recall, precision, label=LABEL_NAMES[c])
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Fine-tuned per-class precision-recall (test)")
ax.legend()
plt.show()

## 8. Calibration — reliability diagram and ECE (fine-tuned)

In [ ]:
ece = expected_calibration_error(ft_probs, y_test)
summary["ece"] = round(ece, 4)
xs, ys = reliability_points(ft_probs, y_test)
print("Expected Calibration Error (ECE): {:.4f}".format(ece))

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "k--", label="perfect")
ax.plot(xs, ys, "o-", label="model")
ax.set_xlabel("Confidence")
ax.set_ylabel("Accuracy")
ax.set_title("Reliability diagram (ECE={:.3f})".format(ece))
ax.legend()
plt.show()

## 9. Misclassified examples (fine-tuned)

In [ ]:
ft_preds = ft_logits.argmax(axis=1)
mis_idx = np.where(ft_preds != y_test)[0][:10]
test_texts = dataset["test"]["text"]
mis = pd.DataFrame(
    {
        "text": [test_texts[i][:80] for i in mis_idx],
        "true": [LABEL_NAMES[y_test[i]] for i in mis_idx],
        "pred": [LABEL_NAMES[ft_preds[i]] for i in mis_idx],
    }
)
print(mis.to_string(index=False))

## 10. Per-class divergence (baseline -> fine-tuned)

In [ ]:
ft_per_class = per_class_f1(y_test, ft_logits.argmax(axis=1))
base_per_class = per_class_f1(y_test, baseline_logits.argmax(axis=1))
ranked = divergent_classes(base_per_class, ft_per_class)
summary["per_class_f1_baseline"] = {k: round(v, 4) for k, v in base_per_class.items()}
summary["per_class_f1_finetuned"] = {k: round(v, 4) for k, v in ft_per_class.items()}
summary["divergent_classes"] = ranked

per_class_table = pd.DataFrame(
    {
        "emotion": LABEL_NAMES,
        "baseline_f1": [round(base_per_class[c], 4) for c in LABEL_NAMES],
        "finetuned_f1": [round(ft_per_class[c], 4) for c in LABEL_NAMES],
    }
)
print(per_class_table.to_string(index=False))
print("\nclasses by |F1 shift| (largest first):", ranked)

## 11. Class-weight ablation (ADR 0012)

In [ ]:
train(output_dir="./outputs/finetuned-model-noweights", use_class_weights=False)
nw_tokenizer = AutoTokenizer.from_pretrained("./outputs/finetuned-model-noweights")
nw_model = AutoModelForSequenceClassification.from_pretrained("./outputs/finetuned-model-noweights")
nw_test_tokenized = tokenize_dataset(DatasetDict({"test": dataset["test"]}), nw_tokenizer)["test"]
nw_logits = predict_split(nw_model, nw_test_tokenized, batch_size=64)
nw_report = evaluation_report(nw_logits, y_test)

delta = ft_report["f1_macro"] - nw_report["f1_macro"]
summary["ablation"] = {
    "with_weights_f1_macro": round(ft_report["f1_macro"], 4),
    "without_weights_f1_macro": round(nw_report["f1_macro"], 4),
    "delta_with_minus_without": round(delta, 4),
}
print("class-weight ablation (test macro F1):")
print("  with weights:    {:.4f}".format(ft_report["f1_macro"]))
print("  without weights: {:.4f}".format(nw_report["f1_macro"]))
print("  delta (with - without): {:+.4f}".format(delta))

del nw_model
if DEVICE == "cuda":
    torch.cuda.empty_cache()

os.makedirs("./outputs", exist_ok=True)
with open("./outputs/nb06_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("\nsaved ./outputs/nb06_summary.json")
print(json.dumps(summary, indent=2))

## Conclusion

_Headline numbers from the executed run. Code outputs are stripped per the repo convention (commit 4afaab2), so these values are the durable record._

- **Frozen-features baseline (test):** accuracy 67.5%, macro F1 0.584.
- **Fine-tuned, balanced class weights (test):** accuracy 92.3%, macro F1 0.887.
- **Macro-F1 gain over baseline:** +51.9% — fine-tuning **clearly exceeds** the frozen-features reference. The gain that collapsed for sentiment (#59, base already TweetEval-tuned) is restored on a task the backbone had not already been trained on.
- **Class-weight ablation (test macro F1):** with weights 0.887 vs without 0.877 (delta +0.010) — balanced weights give a small, consistent gain concentrated in the rare classes.
- **Calibration:** ECE 0.044 (well-calibrated softmax confidences).
- **Per-class divergence (baseline -> fine-tuned), largest first:** love, fear, surprise, anger, sadness, joy — the rare/hard classes gain most (love +0.43, fear +0.33, surprise +0.32 macro-F1).
- **Train imbalance:** `surprise` = 3.57% of train (572/16000, rarest class); `joy` and `sadness` dominate.

See ADR 0011 (task pivot + task-agnostic backbone) and ADR 0012 (balanced class weights). Supersedes the sentiment result frozen at tag `v1-sentiment` (#59).